In [42]:
import numpy as np
import time              #Importa o módulo pra medição de tempo pra usar o time.perf_counter() no k-NN
import matplotlib
matplotlib.use("agg")    # isso é pra poder salvar a figura diretamente como png
import matplotlib.pyplot as plt  # isso eh pra gerar os graficos


In [43]:
from sklearn.datasets import load_iris # funcao pra carregar o conjunto de dados Iris
from sklearn.model_selection import train_test_split, StratifiedKFold
# train_test_split eh uma funcao pra dividir os dados em treino e teste
# StratifiedKFold eh pra usar na validacao cruzada 
# divide o treino em k partes(folds), assegurando que cada partição contenha a mesma distribuição percentual de classes
from sklearn.neighbors import KNeighborsClassifier #implementação oficial do k-NN do scikit-learn, so pra usar no exer8

In [44]:
SEMENTE = 42
np.random.seed(SEMENTE) # Inicializa o gerador de números pseudoaleatórios do NumPy com essa semente
# Garante a reprodutibilidade científica do experimento. 
# Sem fixar a semente, cada execução geraria permutações diferentes nos dados, 
# alterando ligeiramente as amostras de treino e teste, as pontuações da validação cruzada e o valor final do melhor k. 
# Com este valor fixo, qualquer pessoa que execute o caderno obtém exatamente os mesmos resultados e matrizes de confusão

In [45]:
iris = load_iris() # carrega todo o database para a variavel iris
X = iris.data # extrai matriz bidimensional, onde as linhas são os exemplos(amostras) e as colunas sao as caracteristicas (features)
# na iris eh matriz com os 4 atributos
y = iris.target # extrai o vetor unidimensional com as classes verdadeiras (rótulos-alvo)


In [46]:
nomes_atributos = iris.feature_names # pega a lista de nomes dos atributos da iris
nomes_classes = list(iris.target_names) # Converte o vetor de nomes das classes originais numa lista clássica de Python
nomes_classes

[np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

In [47]:
print("Formato de X:", X.shape) # printa as dimensoes da matriz, 150 amostras (linhas)por 4 variáveis/atributos (colunas)
print("Formato de y:", y.shape) # printa o numero 150, sao 150 rótulos/respostas, uma para cada uma das 150 flores
for c, nome in enumerate(nomes_classes): # esse for verifica se o dataset iris eh balanceado
    quantidade = list(y).count(c)
    print(c, nome, quantidade)

Formato de X: (150, 4)
Formato de y: (150,)
0 setosa 50
1 versicolor 50
2 virginica 50


In [48]:
X_treino, X_teste, y_treino, y_test = train_test_split(X, y, test_size = 0.3, random_state=SEMENTE, stratify= y)
# essa funcao divide o conj teste pra ser 30% e 70% o treino
# random_state=SEMENTE embaralha as amostras, pra q a divisao seja sempre exatamente o mesmo em qualquer execução
# stratify= y faz a estratificação nos rótulos verdadeiros, 
# pra garantir proporção idêntica de classes em ambos os lados: 35 flores de cada classe no treino e 15 flores de cada classe no teste

In [49]:
print("Treino:", X_treino.shape, np.bincount(y_treino))
# confirma q o treino teve 105 amostras (4 atributos) e sendo 35 de cada classe(target, rotulo alvo)
print("Teste:", X_teste.shape, np.bincount(y_test))
# confirma q o teste teve 45 amostras (4 atributos) e sendo 15 de cada classe

Treino: (105, 4) [35 35 35]
Teste: (45, 4) [15 15 15]


In [50]:
def ajustar_minmax(X_tr):
    # axis=0, Vertical, Percorre as linhas de cada coluna, da um resultado para cada coluna
    minimo = X_tr.min(axis = 0) # vai retornar o menor numero de cada coluna do treino (um vetor)
    maximo = X_tr.max(axis = 0) # vai retornar o maior numero de cada coluna do treino

    # faixa eh a amplitude, o denominador da funcao minmax
    faixa = np.where(maximo-minimo == 0, 1.0, maximo - minimo)
    # quando max-min for 0 a faixa vai ser 1, se isso nao acontecer a faixa vai ser maximo - minimo
    return minimo, faixa


In [51]:
def aplicar_minmax(X_qualquer, minimo, faixa):# pega uma matriz de dados qualquer(pode ser o conj de treino, teste ou validação) e os vetores minimo e faixa
    return (X_qualquer - minimo)/faixa

# a funcao maxmin em si, o numpy faz operações sem precisar do for

In [52]:
minimo, faixa = ajustar_minmax(X_treino) # pega os menores valores e amplitudes de cada coluna do X_treino
X_treino_n = aplicar_minmax(X_treino, minimo, faixa) # normaliza o treino 
X_teste_n = aplicar_minmax(X_teste, minimo, faixa) # normaliza o teste, reaproveitando os parâmetros (minimo e faixa) extraídos do treino
# Por que não ajustar_minmax(X_teste), para ter uma prevenção de data leakage, 
# porque o modelo tem de avaliar essas novas amostras (o teste) com base estrita nos limites aprendidos durante a fase de treino

In [53]:
n = [1,2,3,4]

In [54]:
max(n)

4

In [55]:
def distancia(A, q, metrica = "euclidiana", p = 3):
    dif  = (A-q)
    if metrica == "euclidiana":
        return np.sqrt(np.sum(dif ** 2, axis = 1))
    elif metrica == "manhattan":
        return np.sum(np.abs(dif), axis = 1)
    elif metrica == "chebyshev":
        return np.max(np.abs(dif), axis = 1)
    raise ValueError(f"Metrica deschonecida: {metrica}")

In [56]:
a_test = np.array([[0.0,0.0], [1.0,1.0], [3.0,4.0]])
q_test = np.array([0.0,0.0])
for metrica in ["euclidiana", "manhattan", "chebyshev"]:
    print(metrica, distancia(a_test, q_test, metrica))

euclidiana [0.         1.41421356 5.        ]
manhattan [0. 2. 7.]
chebyshev [0. 1. 4.]


In [57]:
class KNN:
    def __init__(self, k = 5, metrica = "euclidiana", ponderado = False, p = 3):
        self.K = k
        self.metrica = metrica
        self.ponderado = ponderado
        self.p = p

    def fit(self, X, y):
        self.X_treino = np.asarray(X, dtype=float)
        self.y_treino = np.asanyarray(y)
        self.classes = np.unique(y)
        return self

    def _classificar_um(self, q):
        d = distancia(self.X_treino, q, self.metrica, self.p)
        viz = np.argsort(d)[:self.K]

        if self.ponderado:
            peso = 1.0 / (d[viz] + 1e-10)
        else:
            peso = np.ones(self.K)

        votos = np.zeros(len(self.classes))
        for i, classe in enumerate(self.classes):
            mascara = self.y_treino[viz] == classe
            votos[i] = peso[mascara].sum()

        return self.classes[np.argmax(votos)]

    def predict(self, X):
        X = np.asarray(X, dtype = float)
        predicao = []
        for q in X:
            predicao.append(self._classificar_um(q))
        return np.array(predicao)

In [58]:
X_mini = np.array([[0,0], [0,1], [1,0], [5,5]])
y_mini = np.array([0,0,1,1])
consultas = np.array([[0.1, 0.2], [4.9, 5.1]])

mini = KNN(k=3).fit(X_mini, y_mini)
print(mini.predict(consultas))

[0 1]


In [59]:
def matriz_confusao(y_real, y_pred, n_classes):
    M = np.zeros((n_classes, n_classes), dtype=int)
    for real, pred in zip(y_real, y_pred):
        M[real,pred] += 1
    return M
def relatorio_metricas(M):
    total = M.sum()
    acuracia = 0
    for c in range(M.shape[0]):
        acuracia += M[c,c]

    acuracia = acuracia/total
    
    precisao, revocacao, f1 = [], [], []
    for c in range(M.shape[0]):
        VP = M[c,c]
        FP = M[:,c].sum()-VP
        FN = M[c,:].sum()-VP

        P = VP/(VP + FP) if (VP + FP) > 0 else 0.0
        R = VP/(VP + FN) if (VP + FN) > 0 else 0.0
        F = 2*P*R/(P + R) if (P + R) > 0 else 0.0

        precisao.append(P)
        revocacao.append(R)
        f1.append(F)

    return {
        "VP": VP,
        "FP": FP,
        "FN": FN,
        "acuracia": acuracia,
        "precisao": np.array(precisao),
        "revocacao": np.array(revocacao),
        "f1": np.array(f1),
        "f1_macro": np.mean(f1),
        "acuracia_balanceada": np.sum(revocacao)/len(revocacao)
    }

In [60]:
M_test = np.array([
    [8,1,1],
    [2,6,2],
    [0,1,9]
])
print(relatorio_metricas(M_test))

{'VP': np.int64(9), 'FP': np.int64(3), 'FN': np.int64(1), 'acuracia': np.float64(0.7666666666666667), 'precisao': array([0.8 , 0.75, 0.75]), 'revocacao': array([0.8, 0.6, 0.9]), 'f1': array([0.8       , 0.66666667, 0.81818182]), 'f1_macro': np.float64(0.7616161616161617), 'acuracia_balanceada': np.float64(0.7666666666666666)}


In [61]:
K_ESCOLHIDO = 5

t0 = time.perf_counter()
modelo = KNN(k=K_ESCOLHIDO)
modelo.fit(X_treino_n, y_treino)
tempo_treino = time.perf_counter()-t0

t0 = time.perf_counter()
y_pred = modelo.predict(X_teste_n)
tempo_predicao = time.perf_counter() - t0

M = matriz_confusao(y_test, y_pred, len(nomes_classes))
resultado_simples = relatorio_metricas(M)

modelo_pond = KNN(k=K_ESCOLHIDO, ponderado=True) 
modelo_pond.fit(X_treino_n, y_treino)
y_pred_pond = modelo_pond.predict(X_teste_n)

M_pond = matriz_confusao(y_test, y_pred_pond, len(nomes_classes))
resultado_pond = relatorio_metricas(M_pond)

print("treino (ms): ", 1000*tempo_treino)
print("predicao (ms): ", 1000*tempo_predicao)
print("Simples:", resultado_simples)
print("Ponderado:", resultado_pond)

treino (ms):  0.826900009997189
predicao (ms):  2.910500013967976
Simples: {'VP': np.int64(12), 'FP': np.int64(0), 'FN': np.int64(3), 'acuracia': np.float64(0.9333333333333333), 'precisao': array([1.        , 0.83333333, 1.        ]), 'revocacao': array([1. , 1. , 0.8]), 'f1': array([1.        , 0.90909091, 0.88888889]), 'f1_macro': np.float64(0.9326599326599326), 'acuracia_balanceada': np.float64(0.9333333333333332)}
Ponderado: {'VP': np.int64(12), 'FP': np.int64(0), 'FN': np.int64(3), 'acuracia': np.float64(0.9333333333333333), 'precisao': array([1.        , 0.83333333, 1.        ]), 'revocacao': array([1. , 1. , 0.8]), 'f1': array([1.        , 0.90909091, 0.88888889]), 'f1_macro': np.float64(0.9326599326599326), 'acuracia_balanceada': np.float64(0.9333333333333332)}


In [62]:
referencia = KNeighborsClassifier(n_neighbors=K_ESCOLHIDO, metric="euclidean")
referencia.fit(X_treino_n, y_treino)
y_ref = referencia.predict(X_teste_n)
proporcao_iguais = np.mean(y_ref == y_pred)
print("Acuracia propria: ", np.mean(y_pred == y_test))
print("Acuracia sklearn: ", np.mean(y_ref == y_test))
print("Predicoes identicas: ", proporcao_iguais)

Acuracia propria:  0.9333333333333333
Acuracia sklearn:  0.9333333333333333
Predicoes identicas:  1.0


In [63]:
def cv_acuracia(X, y, k, metrica = "euclidiana", n_folds = 5):
    skf = StratifiedKFold(
        n_splits= n_folds,
        shuffle= True,
        random_state= SEMENTE
    )
    scores = []

    for idx_tr, idx_val in skf.split(X,y):
        minimo_fold, faixa_fold = ajustar_minmax(X[idx_tr])
        Xtr = aplicar_minmax(X[idx_tr], minimo_fold, faixa_fold)
        Xva = aplicar_minmax(X[idx_val], minimo_fold, faixa_fold)  

        modelo_fold = KNN(k=k, metrica=metrica)
        modelo_fold.fit(Xtr, y[idx_tr])
        yva_pred = modelo_fold.predict(Xva)
        scores.append(np.mean(yva_pred == y[idx_val]))

    return np.mean(scores), np.std(scores)

ks = list(range(1,32,2))
resultados_cv = [cv_acuracia(X_treino, y_treino, k) for k in ks]
medias_cv = [resultado[0] for resultado in resultados_cv]
desvios_cv = [resultado[1] for resultado in resultados_cv]
melhor_k = ks[int(np.argmax(medias_cv))]
print("Melhor k:", melhor_k)

for k, media, desvio in zip(ks, medias_cv, desvios_cv):
    print(k, media, desvio)

Melhor k: 3
1 0.9523809523809523 0.030116930096841705
3 0.9714285714285713 0.0380952380952381
5 0.9714285714285713 0.0380952380952381
7 0.9714285714285713 0.0380952380952381
9 0.9714285714285713 0.0380952380952381
11 0.9619047619047618 0.055532875189002885
13 0.9619047619047618 0.055532875189002885
15 0.9619047619047618 0.055532875189002885
17 0.9523809523809523 0.05216405309573012
19 0.9523809523809523 0.06023386019368343
21 0.9333333333333332 0.07126966450997983
23 0.9238095238095237 0.04856209060564558
25 0.9238095238095237 0.038095238095238085
27 0.9047619047619048 0.04259177099999599
29 0.9047619047619048 0.04259177099999599
31 0.9047619047619048 0.06023386019368343


In [ ]:
X2_tr = X_treino_n[:,[2,3]]
modelo_2d = KNN(k = 1, metrica="euclidiana")

modelo_2d.fit(X2_tr, y_treino)

gx, gy = np.meshgrid(
    np.linspace(-0.05, 1.05, 220),
    np.linspace(-0.05, 1.05, 220)
)

grade = np.c_[gx.ravel(), gy.ravel()]
Z = modelo_2d.predict(grade).reshape(gx.shape)

cores = ["#1f4a78", "#c8501e", "#2e8b57"]
marcas = ["o", "s", "^"]

fig, ax = plt.subplots(figsize = (7.2,5.4))
ax.contourf(gx,gy,Z, levels = [-0.5, 0.5, 1.5, 2.5], colors = cores, alpha = 0.2)

for c, nome in enumerate(nomes_classes):
    mascara = y_treino == c
    ax.scatter(
        X2_tr[mascara, 0],
        X2_tr[mascara, 1],
        c=cores[c],
        marker=marcas[c],
        label=nome
    )

ax.set_xlabel("Comprimento da petala (normalizado)")
ax.set_ylabel("Largura da petala (normalizada)")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig("fronteira_knn.png")
plt.close(fig)
